In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
def load_data(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath)
    print(f"Loaded '{filepath}' -- shape: {df.shape}")
    return df

In [3]:
load_data('exampledataset1.csv')

Loaded 'exampledataset1.csv' -- shape: (32, 5)


,Feature 1,Feature 2,Feature 3,Feature 4,Target
0,0.978826,0.269072,0.428748,0.610744,1
1,0.516343,0.242067,0.749536,0.066392,0
2,0.419780,0.294532,0.495886,0.623970,1
3,0.696939,0.712466,0.604261,0.515039,1
4,0.844730,0.153239,0.740513,0.906168,0
5,0.263913,0.223969,0.626643,0.202905,1
6,0.779068,0.102143,0.645494,0.871855,1
7,0.368930,0.781500,0.839945,0.791160,0
8,0.020271,0.780910,0.880310,0.121714,0
9,0.465586,0.153995,0.341292,0.635890,1


In [4]:
df = load_data('exampledataset1.csv')

Loaded 'exampledataset1.csv' -- shape: (32, 5)


In [5]:
def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isnull().sum()
    total_missing = missing.sum()
    if total_missing == 0:
        print("Missing values: none found.")
    else:
        print(f"Missing values detected:\n{missing[missing > 0]}")
        numeric_cols = df.select_dtypes(include="number").columns
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
        print(f"  Filled {total_missing} missing value(s) with column medians.")
    return df

In [6]:
handle_missing_values(df)

Missing values: none found.


,Feature 1,Feature 2,Feature 3,Feature 4,Target
0,0.978826,0.269072,0.428748,0.610744,1
1,0.516343,0.242067,0.749536,0.066392,0
2,0.419780,0.294532,0.495886,0.623970,1
3,0.696939,0.712466,0.604261,0.515039,1
4,0.844730,0.153239,0.740513,0.906168,0
5,0.263913,0.223969,0.626643,0.202905,1
6,0.779068,0.102143,0.645494,0.871855,1
7,0.368930,0.781500,0.839945,0.791160,0
8,0.020271,0.780910,0.880310,0.121714,0
9,0.465586,0.153995,0.341292,0.635890,1


In [7]:
def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    removed = before - len(df)
    if removed == 0:
        print("Duplicates: none found.")
    else:
        print(f"Duplicates: removed {removed} row(s).")
    return df

In [8]:
remove_duplicates(df)

Duplicates: none found.


,Feature 1,Feature 2,Feature 3,Feature 4,Target
0,0.978826,0.269072,0.428748,0.610744,1
1,0.516343,0.242067,0.749536,0.066392,0
2,0.419780,0.294532,0.495886,0.623970,1
3,0.696939,0.712466,0.604261,0.515039,1
4,0.844730,0.153239,0.740513,0.906168,0
5,0.263913,0.223969,0.626643,0.202905,1
6,0.779068,0.102143,0.645494,0.871855,1
7,0.368930,0.781500,0.839945,0.791160,0
8,0.020271,0.780910,0.880310,0.121714,0
9,0.465586,0.153995,0.341292,0.635890,1


In [9]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    print("\n--- Data Preprocessing ---")
    df = handle_missing_values(df)
    df = remove_duplicates(df)
    return df

In [10]:
preprocess(df)


--- Data Preprocessing ---
Missing values: none found.
Duplicates: none found.


,Feature 1,Feature 2,Feature 3,Feature 4,Target
0,0.978826,0.269072,0.428748,0.610744,1
1,0.516343,0.242067,0.749536,0.066392,0
2,0.419780,0.294532,0.495886,0.623970,1
3,0.696939,0.712466,0.604261,0.515039,1
4,0.844730,0.153239,0.740513,0.906168,0
5,0.263913,0.223969,0.626643,0.202905,1
6,0.779068,0.102143,0.645494,0.871855,1
7,0.368930,0.781500,0.839945,0.791160,0
8,0.020271,0.780910,0.880310,0.121714,0
9,0.465586,0.153995,0.341292,0.635890,1


In [11]:
def engineer_features(df: pd.DataFrame, target_col: str = "Target") -> pd.DataFrame:
    print("\n--- Feature Engineering ---")

    feature_cols = [c for c in df.columns if c != target_col]

    # Row-wise statistics
    df["row_mean"]  = df[feature_cols].mean(axis=1)
    df["row_std"]   = df[feature_cols].std(axis=1)
    df["row_range"] = df[feature_cols].max(axis=1) - df[feature_cols].min(axis=1)
    print("  + Row statistics : row_mean, row_std, row_range")

    # Pairwise interaction terms
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            a, b = feature_cols[i], feature_cols[j]
            col = f"{a}_x_{b}".replace(" ", "_")
            df[col] = df[a] * df[b]
    n = len(feature_cols) * (len(feature_cols) - 1) // 2
    print(f"  + Interaction terms : {n} pairwise products")

    # Ratio features
    eps = 1e-9
    for i in range(len(feature_cols) - 1):
        a, b = feature_cols[i], feature_cols[i + 1]
        col = f"{a}_div_{b}".replace(" ", "_")
        df[col] = df[a] / (df[b] + eps)
    print(f"  + Ratio features : {len(feature_cols) - 1} consecutive ratios")

    # Squared terms
    for col in feature_cols:
        df[f"{col}_sq".replace(" ", "_")] = df[col] ** 2
    print(f"  + Squared terms  : {len(feature_cols)} columns")

    print(f"\n  Total columns after engineering: {df.shape[1]}")
    return df

In [12]:
engineer_features(df)


--- Feature Engineering ---
  + Row statistics : row_mean, row_std, row_range
  + Interaction terms : 6 pairwise products
  + Ratio features : 3 consecutive ratios
  + Squared terms  : 4 columns

  Total columns after engineering: 21


,Feature 1,Feature 2,Feature 3,Feature 4,Target,row_mean,row_std,row_range,Feature_1_x_Feature_2,Feature_1_x_Feature_3,...,Feature_2_x_Feature_3,Feature_2_x_Feature_4,Feature_3_x_Feature_4,Feature_1_div_Feature_2,Feature_2_div_Feature_3,Feature_3_div_Feature_4,Feature_1_sq,Feature_2_sq,Feature_3_sq,Feature_4_sq
0,0.978826,0.269072,0.428748,0.610744,1,0.571847,0.305120,0.709754,0.263375,0.419670,...,0.115364,0.164334,0.261855,3.637785,0.627576,0.702009,0.958100,0.072400,0.183825,0.373008
1,0.516343,0.242067,0.749536,0.066392,0,0.393585,0.300989,0.683144,0.124990,0.387018,...,0.181438,0.016071,0.049763,2.133058,0.322956,11.289553,0.266610,0.058596,0.561804,0.004408
2,0.419780,0.294532,0.495886,0.623970,1,0.458542,0.138037,0.329438,0.123639,0.208163,...,0.146054,0.183779,0.309418,1.425244,0.593951,0.794727,0.176215,0.086749,0.245903,0.389339
3,0.696939,0.712466,0.604261,0.515039,1,0.632176,0.091544,0.197427,0.496545,0.421133,...,0.430515,0.366948,0.311218,0.978207,1.179070,1.173233,0.485724,0.507608,0.365131,0.265265
4,0.844730,0.153239,0.740513,0.906168,0,0.661162,0.345450,0.752929,0.129446,0.625534,...,0.113475,0.138860,0.671029,5.512500,0.206936,0.817192,0.713569,0.023482,0.548360,0.821140
5,0.263913,0.223969,0.626643,0.202905,1,0.329357,0.199799,0.423738,0.059108,0.165379,...,0.140349,0.045444,0.127149,1.178346,0.357411,3.088357,0.069650,0.050162,0.392681,0.041170
6,0.779068,0.102143,0.645494,0.871855,1,0.599640,0.344433,0.769712,0.079576,0.502884,...,0.065933,0.089054,0.562777,7.627228,0.158240,0.740369,0.606947,0.010433,0.416663,0.760131
7,0.368930,0.781500,0.839945,0.791160,0,0.695384,0.219134,0.471015,0.288319,0.309881,...,0.656417,0.618292,0.664531,0.472079,0.930418,1.061663,0.136109,0.610742,0.705508,0.625934
8,0.020271,0.780910,0.880310,0.121714,0,0.450801,0.442382,0.860039,0.015830,0.017845,...,0.687443,0.095048,0.107146,0.025958,0.887085,7.232611,0.000411,0.609820,0.774946,0.014814
9,0.465586,0.153995,0.341292,0.635890,1,0.399191,0.203231,0.481895,0.071698,0.158901,...,0.052557,0.097924,0.217024,3.023384,0.451212,0.536715,0.216770,0.023714,0.116480,0.404356


In [13]:
def train_and_evaluate(df: pd.DataFrame, target_col: str = "Target",
                       test_size: float = 0.2, random_state: int = 42):
    # Split into features and target
    feature_cols = [c for c in df.columns if c != target_col]
    X = df[feature_cols]
    y = df[target_col]

    print(f"\n--- Feature/Target Split ---")
    print(f"  Features : {X.shape[1]} columns")
    print(f"  Target   : '{target_col}' -- classes: {sorted(y.unique())}")

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    print(f"\n--- Train/Test Split ---")
    print(f"  Training samples : {len(X_train)}")
    print(f"  Testing samples  : {len(X_test)}")

    # Train model
    model = LogisticRegression(random_state=random_state, max_iter=1000)
    model.fit(X_train, y_train)
    print("\n--- Model Training ---")
    print("  Model: LogisticRegression (max_iter=1000)")

    # Predict and evaluate
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n--- Results ---")
    print(f"  Accuracy Score: {accuracy:.4f}")

    return model, accuracy

In [14]:
train_and_evaluate(df)


--- Feature/Target Split ---
  Features : 20 columns
  Target   : 'Target' -- classes: [np.int64(0), np.int64(1)]

--- Train/Test Split ---
  Training samples : 25
  Testing samples  : 7

--- Model Training ---
  Model: LogisticRegression (max_iter=1000)

--- Results ---
  Accuracy Score: 0.4286


(LogisticRegression(max_iter=1000, random_state=42), 0.42857142857142855)

## MNIST Neural Network — Image Classification

In [15]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow info/warning messages

import tensorflow as tf
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Flatten
from sklearn.model_selection import train_test_split



In [16]:
# -------------------------------------------------------------------
# 1. Load the MNIST dataset
# -------------------------------------------------------------------
(X_train_full, y_train_full), (X_test, y_test) = mnist.load_data()

# -------------------------------------------------------------------
# 2. Preprocess: normalize pixel values from [0, 255] to [0, 1]
# -------------------------------------------------------------------
X_train_full = X_train_full / 255.0
X_test        = X_test        / 255.0

# -------------------------------------------------------------------
# 3. Split training data into training and validation sets
# -------------------------------------------------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)
print(f"Training samples   : {len(X_train)}")
print(f"Validation samples : {len(X_val)}")
print(f"Test samples       : {len(X_test)}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 7s 1us/step
Training samples   : 48000
Validation samples : 12000
Test samples       : 10000


In [17]:
# -------------------------------------------------------------------
# 4. Define the neural network architecture
# -------------------------------------------------------------------
model = Sequential([
    Flatten(input_shape=(28, 28)),   # Flatten 28x28 images to a 784-element vector
    Dense(128, activation='relu'),   # Hidden layer 1: 128 neurons, ReLU activation
    Dense(64,  activation='relu'),   # Hidden layer 2: 64 neurons, ReLU activation
    Dense(10,  activation='softmax') # Output layer: 10 classes (digits 0-9)
])

model.summary()

c:\Users\THISPC\Documents\ROMTech\Probability\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# -------------------------------------------------------------------
# 5. Compile the model
# -------------------------------------------------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# -------------------------------------------------------------------
# 6. Train the model
# -------------------------------------------------------------------
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=1
)

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9237 - loss: 0.2672 - val_accuracy: 0.9554 - val_loss: 0.1455
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9671 - loss: 0.1126 - val_accuracy: 0.9694 - val_loss: 0.1016
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9766 - loss: 0.0773 - val_accuracy: 0.9688 - val_loss: 0.1086
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9810 - loss: 0.0599 - val_accuracy: 0.9718 - val_loss: 0.0977
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9851 - loss: 0.0464 - val_accuracy: 0.9768 - val_loss: 0.0811
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9881 - loss: 0.0354 - val_accuracy: 0.9770 - val_loss: 0.0899
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9895 - loss: 0.0320 - val_accuracy: 0.9747 - val_loss: 0.0987
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9920 - loss: 0.0230 - 

In [19]:
# -------------------------------------------------------------------
# 7. Evaluate the model on the test set
# -------------------------------------------------------------------
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Test Loss:     0.1226
Test Accuracy: 0.9686
